In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# create the semesters
survey = pd.read_csv("BasicSurvey.csv", low_memory=False)
fmt = "%d%b%Y %H:%M:%S"

confini = []
for w in range(1, 9):
    date_w = pd.to_datetime(survey[f"StartDate_{w}"], format=fmt, errors="coerce")
    confini.append(date_w.median().normalize())

for w, c in enumerate(confini, start=1):
    print(f"semester {w} starts on {c.date()}")

semester 1 starts on 2015-08-07
semester 2 starts on 2016-02-07
semester 3 starts on 2016-06-22
semester 4 starts on 2016-11-04
semester 5 starts on 2017-04-12
semester 6 starts on 2017-11-13
semester 7 starts on 2018-04-29
semester 8 starts on 2019-04-16


In [ ]:
df = pd.read_csv("FitbitActivity.csv", low_memory=False, parse_dates=["datadate"])

# create the intervals
bins = [df["datadate"].min()] + confini[1:] + [df["datadate"].max() + pd.Timedelta("1D")]

# assign each date to a semester
df["semester"] = pd.cut(df["datadate"], bins=bins, labels=range(1, 9), right=False)

# Filter: compliance ≥ 80% and only days within a semester
d = df[(df["complypercent"] >= 80) & df["semester"].notna()]

# mean of fatburncal per person per semester
per_person = d.groupby(["egoid", "semester"], observed=True)["fatburncal"].mean()

# final table for visualization
table = per_person.unstack("semester").round(1)

days = d.groupby(["egoid", "semester"], observed=True).size().unstack("semester")
table = table.where(days >= 10)

print(table.head())
print(table.shape)

semester       1       2       3       4       5       6      7   8
egoid                                                              
10237      962.5   882.3   840.7  1172.7  1308.4  1262.0  833.3 NaN
10469        NaN     NaN   447.1   636.2   573.9     NaN    NaN NaN
10547     1167.0     NaN     NaN     NaN     NaN     NaN    NaN NaN
11002     1633.2  1296.9  1184.5   805.0   965.2  1055.8    NaN NaN
11128      685.5   566.0   190.7     NaN     NaN     NaN    NaN NaN
(618, 8)


In [25]:
# to see all the table
pd.set_option("display.max_rows", None)      # nessun limite alle righe
pd.set_option("display.max_columns", None)   # nessun limite alle colonne
pd.set_option("display.width", None)         # non va a capo con le colonne

table

semester,1,2,3,4,5,6,7,8
egoid,,,,,,,,
10237,962.5,882.3,840.7,1172.7,1308.4,1262.0,833.3,NaN
10469,NaN,NaN,447.1,636.2,573.9,NaN,NaN,NaN
10547,1167.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11002,1633.2,1296.9,1184.5,805.0,965.2,1055.8,NaN,NaN
11128,685.5,566.0,190.7,NaN,NaN,NaN,NaN,NaN
11148,634.3,670.2,NaN,NaN,NaN,NaN,NaN,NaN
11402,1109.0,733.2,1038.7,998.4,1032.1,1037.5,1347.2,NaN
11482,559.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11910,713.0,984.3,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
print(table.notna().sum())        # people per semester
print(df["datadate"].max())         # last day Fitbit

semester
1    470
2    409
3    433
4    380
5    363
6    321
7    238
8    101
dtype: int64
2019-05-19 00:00:00
